$$dX_t=-\nabla V(X_T)dt+BdW_t$$

$$X_t=\begin{pmatrix} x_t \\ y_t \end{pmatrix}$$

$$V(X)=\frac{1}{2}m_x^2 x^2 + \frac{1}{2}m_y^2 y^2 + m_{xy}^2xy + \frac{1}{4}\lambda_x x^4 + \frac{1}{4}\lambda_y y^4$$

$$B=\begin{pmatrix}\sigma_x & 0 \\ 0 & \sigma_y \end{pmatrix}$$

In [17]:
using LinearAlgebra
using Statistics
using DifferentialEquations
using Plots
using Random
using DataFrames
using Dates
using CSV

In [18]:
m2x = 1.0
m2y = 1.0
m2xy = 0.5
lambdax = 0.5
lambday = 0.5
sigmax = 0.5
sigmay = 0.5
x0 = 1.0
y0 = 0.5
tEnd = 0.5

0.5

In [19]:
A = [m2x m2xy; m2xy m2y] .* (-1)
B = [sigmax 0.0; 0.0 sigmay]
vecB = [sigmax, sigmay]
F = [lambdax 0.0 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0 0.0 lambday] .* (-1)
XIni = [x0, y0]

2-element Vector{Float64}:
 1.0
 0.5

In [20]:
function drift!(driftVec, u, p, t)
    copyto!(driftVec, A * u + F * kron(u, u, u))
end

function vol!(volVec, u, p, t)
    copyto!(volVec, vecB)
end

vol! (generic function with 1 method)

Solve original SDE

In [ ]:
Random.seed!(1)
discScheme = EM()
dt = 0.0001
NPath = 1E7
NGrp = 10000
NPathPerGrp = (Int)(NPath / NGrp)
XNormMax = 0.0
outFile = "xTyT.csv"
outDf = DataFrame([Symbol(cn) => Float64[] for cn in ["x", "y"]])

for iGrp in 1:NGrp
    sdeProb = SDEProblem(drift!, vol!, XIni, (0, tEnd))
    paths = solve(EnsembleProblem(sdeProb), discScheme, EnsembleThreads(), trajectories=NPathPerGrp, dt=dt, adaptive=false)
    for p in paths
        push!(outDf, [p.u[end][1], p.u[end][2]])
        XNormMax = max(XNormMax, maximum([norm(u) for u in p.u]))
    end
    NPathGenerated = iGrp * NPathPerGrp
    CSV.write(outFile, outDf[(NPathGenerated - NPathPerGrp + 1):NPathGenerated,:], append=true)
    println(string(iGrp * NPathPerGrp) * " paths generated:" * string(now())); flush(stdout)
end

In [22]:
println(XNormMax)

1.8305451964833102


In [23]:
xEnds = outDf[:,1]
yEnds = outDf[:,2]
meanx = mean(xEnds)
meany = mean(yEnds)
stderrx = std(xEnds) / sqrt(NPath)
stderry = std(yEnds) / sqrt(NPath)
maxx = maximum(xEnds)
maxy = maximum(yEnds)
println(meanx, " ", stderrx, " ", maxx)
println(meany, " ", stderry, " ", maxy)

0.467512813357783 8.049611277711047e-5 1.664133896578412
0.15861240017278086 8.694831053757536e-5 1.4045724480996282
